In [0]:

df = spark.sql("select * from gizmobox.bronze.py_customers")
# Another way
#df = spark.table('gizmobox.bronze.py_customers')
display(df)

In [0]:
df.printSchema()

In [0]:
# cast columns to appropriate types

from pyspark.sql.functions import *
df = df.select(
    col('created_timestamp').cast('timestamp'),
    col('customer_id').cast('long'),
    col('customer_name').cast('string'),
    col('date_of_birth').cast('date'),
    col("email").cast('string'),
    col('member_since').cast('date'),
    col('telephone').cast('string')
)
df.printSchema()

In [0]:
#check null counts in customer_id
df.groupby('customer_id').count().filter(col('customer_id').isNull()).show()

In [0]:
# Remove records with null customer id
df = df.dropna(subset=['customer_id']).dropDuplicates()
df.groupby('customer_id').count().filter(col('customer_id').isNull()).show()


In [0]:
# Remove duplicate records based on created timestamp

from pyspark.sql.window import *
from pyspark.sql.functions import *

window_spec = Window.partitionBy('customer_id').orderBy(desc('created_timestamp'))

# duplicates before
df.groupby('customer_id').count().filter(col('count')>1).show()
# duplicates after
df = df.withColumn('rank', rank().over(window_spec)).filter(col('rank') == 1).drop('rank')
df.groupby('customer_id').count().filter(col('count')>1).show()

In [0]:
# writing df to silver
df.writeTo("gizmobox.silver.py_customers").createOrReplace()

In [0]:
%sql
-- add not null to primary key
Alter table gizmobox.silver.py_customers
alter column customer_id set not null;

In [0]:
%sql
select * from gizmobox.silver.py_customers